# Automatic Sleep Scoring Comparison

Compare automatic sleep stagers (**YASA**, **GSSC**, **SleepFM**, and SleepFM recalibrations) against a manual hypnogram.

---
1. Select your **EDF folder** and **scoring folder**
2. Confirm the **channel mapping**
3. Pick which **stagers** to run
4. View **comparison plots** (hypnodensities, confusion matrices, agreement metrics)

_Launch this notebook with Voila: `bash autoscore.sh` or `voila score_edf_voila.ipynb`_

In [ ]:
#%% Imports and setup
import sys
import os
import time
import warnings
import traceback
from pathlib import Path
from contextlib import redirect_stderr, redirect_stdout
import io as _io

os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
mne.set_log_level('ERROR')
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from ipyfilechooser import FileChooser
    HAS_FILECHOOSER = True
except ImportError:
    HAS_FILECHOOSER = False

# Add local scoredf package
NOTEBOOK_DIR = Path.cwd() if '__file__' not in dir() else Path(__file__).parent
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from scoredf.io_score import detect_and_read_scoring, match_files, window_from_scoring
from scoredf.channels import (
    detect_channels, get_edf_channel_names, check_stager_requirements,
    channel_mapping, ALIASES,
)
from scoredf.plots import (
    plot_hypnodensities, plot_confusion_matrices, compute_agreement,
    plot_agreement_summary, plot_hypnogram_grid, STAGES, STAGE_COLORS, STAGER_COLORS,
)

from sleephd.io import NapWindow, STAGE_NAMES
from sleephd.alignment import (
    build_epoch_table, aggregate_hypnodensity, make_nap_epochs, align_manual_hypnogram,
)
from sleephd.signals import EEG_DERIVATIONS

# Probe stager availability
GSSC_AVAILABLE = False
try:
    from gssc.infer import EEGInfer  # noqa: F401
    GSSC_AVAILABLE = True
except ImportError:
    pass

YASA_AVAILABLE = False
try:
    import yasa  # noqa: F401
    YASA_AVAILABLE = True
except ImportError:
    pass

SLEEPFM_AVAILABLE = False
try:
    from sleephd.sleepfm import _default_sleepfm_root
    SLEEPFM_AVAILABLE = (_default_sleepfm_root() / 'sleepfm' / 'checkpoints').exists()
except Exception:
    pass

ALL_STAGERS = ['YASA', 'GSSC', 'SleepFM', 'SleepFM recal A', 'SleepFM recal B']

# Global state
STATE = {}

print('Packages loaded.')
avail = []
if YASA_AVAILABLE: avail.append('YASA')
if GSSC_AVAILABLE: avail.append('GSSC')
if SLEEPFM_AVAILABLE: avail.append('SleepFM')
print(f'Available stagers: {avail if avail else "none (install packages in sleephd env)"}')

#%% ============================================================
#    PHASE 1: Folder selection and file scanning
#    ============================================================
display(HTML('<hr style="height:4px; background-color:black; border:none;"><h2>1. Select your data folders</h2>'))

if HAS_FILECHOOSER:
    chooser_edf = FileChooser(os.getcwd())
    chooser_edf.title = '<b>EDF folder</b>'
    chooser_edf.show_only_dirs = True

    chooser_scoring = FileChooser(os.getcwd())
    chooser_scoring.title = '<b>Scoring folder</b>'
    chooser_scoring.show_only_dirs = True
else:
    chooser_edf = widgets.Text(description='EDF folder:', layout=widgets.Layout(width='80%'))
    chooser_scoring = widgets.Text(description='Scoring folder:', layout=widgets.Layout(width='80%'))

scan_button = widgets.Button(description='Scan folders', button_style='success', icon='search')
scan_output = widgets.Output()

# Phase 2 widgets (created after scan)
phase2_box = widgets.VBox()
run_button = widgets.Button(description='Run scoring', button_style='primary', icon='play')
run_output = widgets.Output()
results_output = widgets.Output()


def get_folder(chooser):
    if HAS_FILECHOOSER:
        return chooser.selected_path or chooser.value
    return chooser.value.strip()


def on_scan_click(_):
    scan_output.clear_output()
    with scan_output:
        edf_folder = get_folder(chooser_edf)
        scoring_folder = get_folder(chooser_scoring)
        if not edf_folder or not Path(edf_folder).is_dir():
            print('Please select a valid EDF folder.')
            return
        if not scoring_folder or not Path(scoring_folder).is_dir():
            print('Please select a valid scoring folder.')
            return

        print(f'EDF folder:     {edf_folder}')
        print(f'Scoring folder: {scoring_folder}')
        print()

        # Match files
        matched, unmatched_edfs, unmatched_scorings = match_files(edf_folder, scoring_folder)
        STATE['matched'] = matched
        STATE['edf_folder'] = edf_folder
        STATE['scoring_folder'] = scoring_folder

        if not matched:
            print('No EDF-scoring matches found. Check that filenames share a common stem.')
            return

        print(f'Matched recordings: {len(matched)}')
        for m in matched:
            print(f'  {m["stem"]:20s}  EDF: {m["edf"].name:30s}  Scoring: {m["scoring"].name}')
        if unmatched_edfs:
            print(f'\nUnmatched EDFs ({len(unmatched_edfs)}):')
            for f in unmatched_edfs:
                print(f'  {f.name}')
        if unmatched_scorings:
            print(f'\nUnmatched scoring files ({len(unmatched_scorings)}):')
            for f in unmatched_scorings:
                print(f'  {f.name}')

        # Auto-detect channels from first EDF
        print('\nDetecting channels from first EDF...')
        first_edf = matched[0]['edf']
        all_ch_names = get_edf_channel_names(first_edf)
        auto_mapping = detect_channels(first_edf)
        STATE['all_ch_names'] = all_ch_names
        STATE['auto_mapping'] = auto_mapping

        detected = {k: v for k, v in auto_mapping.items() if v is not None}
        missing = {k for k, v in auto_mapping.items() if v is None}
        print(f'Detected {len(detected)}/{len(auto_mapping)} channels: {detected}')
        if missing:
            print(f'Not found: {sorted(missing)}')

    # Build Phase 2 UI
    _build_phase2(auto_mapping, all_ch_names, matched)


def _build_phase2(auto_mapping, all_ch_names, matched):
    # Channel mapping dropdowns
    options = ['-- not available --'] + sorted(all_ch_names)
    dropdowns = {}
    channel_rows = []
    for standard in ALIASES:
        current = auto_mapping.get(standard)
        dd = widgets.Dropdown(
            options=options,
            value=current if current in all_ch_names else '-- not available --',
            layout=widgets.Layout(width='200px'),
        )
        dropdowns[standard] = dd
        channel_rows.append(
            widgets.HBox([
                widgets.Label(f'{standard}:', layout=widgets.Layout(width='100px')),
                dd,
            ])
        )
    STATE['dropdowns'] = dropdowns

    channel_box = widgets.VBox(
        [widgets.HTML('<h3>Channel mapping</h3><p>Adjust if needed:</p>')] + channel_rows,
    )

    # Stager checkboxes
    checkboxes = {}
    stager_rows = []
    n_recordings = len(matched)

    for name in ALL_STAGERS:
        enabled = True
        note = ''

        if name == 'YASA' and not YASA_AVAILABLE:
            enabled = False
            note = ' (yasa package not installed)'
        elif name == 'GSSC' and not GSSC_AVAILABLE:
            enabled = False
            note = ' (gssc package not installed)'
        elif name.startswith('SleepFM') and not SLEEPFM_AVAILABLE:
            enabled = False
            note = ' (SleepFM checkpoints not found)'

        if name == 'SleepFM recal A' and n_recordings < 2:
            enabled = False
            note = f' (needs >= 2 recordings, have {n_recordings})'
        if name == 'SleepFM recal B' and n_recordings < 3:
            enabled = False
            note = f' (needs >= 3 recordings, have {n_recordings})'

        cb = widgets.Checkbox(
            value=enabled,
            description=name + note,
            disabled=not enabled,
            indent=False,
            layout=widgets.Layout(width='400px'),
        )
        checkboxes[name] = cb
        stager_rows.append(cb)

    STATE['checkboxes'] = checkboxes

    if n_recordings < 6 and SLEEPFM_AVAILABLE:
        stager_rows.append(widgets.HTML(
            f'<p style="color:orange;">Note: recal B works best with >= 6 recordings (you have {n_recordings}).</p>'
        ))

    stager_box = widgets.VBox(
        [widgets.HTML('<h3>Stagers to run</h3>')] + stager_rows,
    )

    # Force recompute checkbox
    force_cb = widgets.Checkbox(value=False, description='Force recompute (ignore cache)', indent=False)
    STATE['force_cb'] = force_cb

    phase2_box.children = [
        widgets.HTML('<hr style="height:4px; background-color:black; border:none;"><h2>2. Confirm channels and select stagers</h2>'),
        widgets.HBox([channel_box, widgets.HTML('&nbsp;' * 10), stager_box]),
        force_cb,
        run_button,
        run_output,
        results_output,
    ]


scan_button.on_click(on_scan_click)
display(chooser_edf, chooser_scoring, scan_button, scan_output, phase2_box)


#%% ============================================================
#    PHASE 3: Run stagers
#    ============================================================
def _get_user_mapping():
    mapping = {}
    for standard, dd in STATE['dropdowns'].items():
        val = dd.value
        mapping[standard] = val if val != '-- not available --' else None
    return mapping


def on_run_click(_):
    run_output.clear_output()
    results_output.clear_output()

    with run_output:
        matched = STATE.get('matched', [])
        if not matched:
            print('No matched recordings. Scan folders first.')
            return

        user_mapping = _get_user_mapping()
        selected_stagers = [name for name, cb in STATE['checkboxes'].items() if cb.value]
        force = STATE['force_cb'].value

        if not selected_stagers:
            print('No stagers selected.')
            return

        # Check channel requirements for selected stagers
        for name in list(selected_stagers):
            ok, missing = check_stager_requirements(user_mapping, name)
            if not ok:
                print(f'{name}: missing channels {missing} -- skipping.')
                selected_stagers.remove(name)
        if not selected_stagers:
            print('No stagers can run with the current channel mapping.')
            return

        # Determine output directory
        out_dir = Path(STATE['edf_folder']).parent / 'autoscore_outputs'
        cache_dir = out_dir / 'cache'
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f'Output directory: {out_dir}')
        print(f'Selected stagers: {selected_stagers}')
        print(f'Recordings: {len(matched)}')
        print()

        # Progress
        progress = widgets.IntProgress(
            min=0, max=len(matched), description='Scoring:', bar_style='info',
            layout=widgets.Layout(width='80%'),
        )
        display(progress)

        # Per-recording data
        five_second = {name: {} for name in selected_stagers}
        recordings_info = {}
        all_frames = []
        failed = []

        for rec_idx, rec in enumerate(matched):
            sid = rec['stem']
            edf_path = rec['edf']
            scoring_path = rec['scoring']
            progress.value = rec_idx
            progress.description = f'{sid}:'
            print(f'--- {sid} ({rec_idx+1}/{len(matched)}) ---')

            try:
                # Read scoring
                hypnogram, epoch_seconds = detect_and_read_scoring(scoring_path)
                print(f'  Scoring: {scoring_path.name} ({len(hypnogram)} epochs, {epoch_seconds}s)')

                # Derive window
                window = window_from_scoring(hypnogram, edf_path, epoch_seconds)
                print(f'  Window: {window.lights_off_clock} -- {window.reveil_clock} '
                      f'({window.duration_seconds/60:.1f} min)')

                recordings_info[sid] = {
                    'edf': edf_path, 'window': window,
                    'hypnogram': hypnogram, 'epoch_seconds': epoch_seconds,
                }

                # Detect channels for this specific EDF
                rec_mapping = detect_channels(edf_path)
                # Override with user choices where the user explicitly set something
                for standard, val in user_mapping.items():
                    if val is not None:
                        rec_mapping[standard] = val

                with channel_mapping(rec_mapping):
                    # --- YASA ---
                    if 'YASA' in selected_stagers:
                        ok, _ = check_stager_requirements(rec_mapping, 'YASA')
                        if ok:
                            print('  Running YASA...')
                            from sleephd.stagers import infer_yasa
                            five_second['YASA'][sid] = infer_yasa(
                                edf_path, window,
                                cache_path=cache_dir / f'{sid}_yasa.csv',
                                force=force,
                            )
                        else:
                            print(f'  YASA: skipped (missing channels for {sid})')

                    # --- GSSC ---
                    if 'GSSC' in selected_stagers:
                        ok, _ = check_stager_requirements(rec_mapping, 'GSSC')
                        if ok:
                            print('  Running GSSC...')
                            from sleephd.stagers import infer_gssc
                            with redirect_stderr(_io.StringIO()):
                                five_second['GSSC'][sid] = infer_gssc(
                                    edf_path, window,
                                    cache_path=cache_dir / f'{sid}_gssc.csv',
                                    force=force,
                                )
                        else:
                            print(f'  GSSC: skipped (missing channels for {sid})')

                    # --- SleepFM ---
                    needs_sleepfm = any(
                        s in selected_stagers
                        for s in ['SleepFM', 'SleepFM recal A', 'SleepFM recal B']
                    )
                    if needs_sleepfm:
                        ok, _ = check_stager_requirements(rec_mapping, 'SleepFM')
                        if ok:
                            print('  Running SleepFM...')
                            from sleephd.signals import infer_sleepfm_montage
                            sfm_result = infer_sleepfm_montage(
                                edf_path, cache_dir,
                                eeg_derivations=list(EEG_DERIVATIONS),
                                cache_tag='aasm_all_eeg',
                                include_eog=True,
                                device='cpu',
                                cover_tail=True,
                                force=force,
                            )
                            if 'SleepFM' in selected_stagers:
                                five_second['SleepFM'][sid] = sfm_result
                            # Store for recalibration
                            recordings_info[sid]['sleepfm_5s'] = sfm_result
                        else:
                            print(f'  SleepFM: skipped (missing channels for {sid})')

            except Exception as e:
                print(f'  ERROR: {e}')
                traceback.print_exc()
                failed.append((sid, str(e)))

        progress.value = len(matched)
        progress.description = 'Done:'
        progress.bar_style = 'success'

        # --- Recalibration (LOSO) ---
        recal_subjects = [
            sid for sid in recordings_info
            if 'sleepfm_5s' in recordings_info[sid]
        ]
        needs_recal = any(
            s in selected_stagers for s in ['SleepFM recal A', 'SleepFM recal B']
        )
        base_recs = None
        if needs_recal and len(recal_subjects) >= 2:
            from sleephd import recalibration
            base_recs = []
            for sid in recal_subjects:
                info = recordings_info[sid]
                base_recs.append(recalibration.prepare_recording(
                    sid, info['sleepfm_5s'], info['window'], info['hypnogram'], info['epoch_seconds'],
                ))

        if 'SleepFM recal A' in selected_stagers and base_recs is not None and len(base_recs) >= 2:
            print('\n--- Recalibration A (LOSO) ---')
            five_second['SleepFM recal A'] = {}
            for held_out in base_recs:
                train = [r for r in base_recs if r.subject != held_out.subject]
                params = recalibration.fit(train)
                five_second['SleepFM recal A'][held_out.subject] = (
                    recalibration.recalibrated_five_second(held_out, params)
                )
                print(f'  {held_out.subject}: T={params["temperature"]:.3f}, converged={params["converged"]}')

        if 'SleepFM recal B' in selected_stagers and base_recs is not None and len(base_recs) >= 3:
            print('\n--- Recalibration B (LOSO, refit fc) ---')
            from sleephd.sleepfm import head_features as _head_features
            from scipy.special import softmax

            # Extract frozen features
            heads = []
            fc_weight = fc_bias = None
            for rec_base in base_recs:
                sid = rec_base.subject
                rec_mapping = detect_channels(recordings_info[sid]['edf'])
                for standard, val in user_mapping.items():
                    if val is not None:
                        rec_mapping[standard] = val

                feat_path = cache_dir / f'{sid}_head_features.npz'
                if not feat_path.exists() or force:
                    with channel_mapping(rec_mapping):
                        primary = _head_features(
                            cache_dir / f'{sid}_aasm_all_eeg_128hz.hdf5'
                        )
                    features = primary['features']
                    extra = len(rec_base.five_second) - len(features)
                    if extra > 0:
                        tail_path = cache_dir / f'{sid}_aasm_all_eeg_tail_128hz.hdf5'
                        if tail_path.exists():
                            with channel_mapping(rec_mapping):
                                tail = _head_features(tail_path)
                            features = np.vstack([features, tail['features'][-extra:]])
                    np.savez(feat_path, features=features,
                             fc_weight=primary['fc_weight'], fc_bias=primary['fc_bias'])

                data = np.load(feat_path)
                fc_weight = data['fc_weight']
                fc_bias = data['fc_bias']
                heads.append(recalibration.prepare_head_recording(rec_base, data['features']))
                print(f'  {sid}: {data["features"].shape[0]} feature steps')

            five_second['SleepFM recal B'] = {}
            inner_folds = min(5, len(heads) - 1) if len(heads) > 2 else len(heads) - 1
            for held_out in heads:
                train = [h for h in heads if h.subject != held_out.subject]
                result = recalibration.fit_fc_nested(
                    train, fc_weight, fc_bias, inner_folds=max(2, inner_folds),
                )
                five_second['SleepFM recal B'][held_out.subject] = (
                    recalibration.refit_five_second(held_out, result['weight'], result['bias'])
                )
                print(f'  {held_out.subject}: lambda={result["penalty"]}')

        # --- Build unified epoch table ---
        print('\nBuilding epoch tables...')
        active_stagers = [s for s in selected_stagers if five_second.get(s)]
        frames = []
        for name in active_stagers:
            for sid in five_second[name]:
                info = recordings_info[sid]
                table = build_epoch_table(
                    info['window'], None, info['hypnogram'],
                    hypnodensity_5s=five_second[name][sid],
                    epoch_seconds=info['epoch_seconds'],
                )
                table['minutes_from_start'] = (
                    (table['start_seconds'] - info['window'].lights_off_seconds) / 60
                )
                frames.append(table.assign(subject=sid, stager=name))

        if not frames:
            print('No results to display.')
            return

        epochs_df = pd.concat(frames, ignore_index=True)
        scored_df = epochs_df.loc[epochs_df['manual_stage'] != 'Unknown']
        subjects = sorted(recordings_info.keys())

        # Save
        epochs_df.to_csv(out_dir / 'epochs_all_stagers.csv', index=False)
        print(f'Saved epoch table to {out_dir / "epochs_all_stagers.csv"}')

        STATE['epochs_df'] = epochs_df
        STATE['scored_df'] = scored_df
        STATE['active_stagers'] = active_stagers
        STATE['subjects'] = subjects
        STATE['out_dir'] = out_dir

        if failed:
            print(f'\nFailed recordings ({len(failed)}):')
            for sid, err in failed:
                print(f'  {sid}: {err}')

        print('\nDone! Generating results...')

    # Generate results
    _show_results()


#%% ============================================================
#    PHASE 4: Results display
#    ============================================================
def _show_results():
    results_output.clear_output()
    with results_output:
        epochs_df = STATE['epochs_df']
        scored_df = STATE['scored_df']
        stagers = STATE['active_stagers']
        subjects = STATE['subjects']
        out_dir = STATE['out_dir']

        display(HTML('<hr style="height:4px; background-color:black; border:none;"><h2>3. Results</h2>'))

        # --- Agreement table ---
        display(HTML('<h3>3.1 Agreement with manual hypnogram</h3>'))
        agreement = compute_agreement(scored_df, stagers, subjects)
        agreement.to_csv(out_dir / 'agreement_summary.csv')
        display(agreement.round(3).style.background_gradient(cmap='Greens', subset=['accuracy', 'kappa_pooled']))

        # --- Agreement figure ---
        try:
            fig = plot_agreement_summary(agreement, stagers, subjects)
            fig.savefig(out_dir / 'agreement_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Agreement plot error: {e}')

        # --- Confusion matrices ---
        display(HTML('<h3>3.2 Confusion matrices</h3>'))
        try:
            fig = plot_confusion_matrices(scored_df, stagers)
            fig.savefig(out_dir / 'confusion_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Confusion plot error: {e}')

        # --- Hypnogram grid ---
        display(HTML('<h3>3.3 Hypnogram comparison</h3>'))
        try:
            fig = plot_hypnogram_grid(epochs_df, stagers, subjects)
            fig.savefig(out_dir / 'hypnograms_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Hypnogram grid error: {e}')

        # --- Per-recording hypnodensities ---
        display(HTML('<h3>3.4 Hypnodensities per recording</h3>'))
        for sid in subjects:
            try:
                fig = plot_hypnodensities(epochs_df, stagers, sid)
                fig.savefig(out_dir / f'hypnodensities_{sid}.png', dpi=130, bbox_inches='tight')
                display(fig)
                plt.close(fig)
            except Exception as e:
                print(f'Hypnodensity plot error for {sid}: {e}')

        display(HTML(f'<hr><p>All outputs saved to <code>{out_dir}</code></p>'))


run_button.on_click(on_run_click)
